In [ ]:

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup # We can still use the scheduler

import numpy as np
from tqdm import tqdm
import os
import matplotlib.pyplot as plt

# --- Import and check for Weights & Biases ---
try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    print("Warning: wandb library not found. Disabling logging. Please pip install wandb.")
    WANDB_AVAILABLE = False

# =================================================================
# 1. CONFIGURATION
# =================================================================
hyperparameter_config = {
    "model_type": "Linear_2nd_Order", # For logging
    "data_file_path": "data_tensor_2.pt",
    "epochs": 200, # Linear models often train faster and can use more epochs
    "batch_size": 2048, # Can often use a larger batch size
    "learning_rate": 1e-5, # Linear models often tolerate a higher learning rate
    "warmup_ratio": 0.1,
    "train_split_ratio": 0.9,
    "project_name": "moral-reasoning-models", # A more general project name
    "run_name": f"linear_2nd_order_{int(np.random.rand()*1000)}",
}

# --- Data & Vocabulary ---
FEATURES = [
    'Intervention', 'Barrier', 'CrossingSignal', 'Man', 'Woman', 
    'Pregnant', 'Stroller', 'OldMan', 'OldWoman', 'Boy', 'Girl', 'Homeless', 
    'LargeWoman', 'LargeMan', 'Criminal', 'MaleExecutive', 'FemaleExecutive', 
    'FemaleAthlete', 'MaleAthlete', 'FemaleDoctor', 'MaleDoctor', 'Dog', 'Cat'
]
BASE_NUM_FEATURES = len(FEATURES)
NUM_FEATURES = BASE_NUM_FEATURES + BASE_NUM_FEATURES**2

# --- System Setup ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def add_second_order_features(data_numpy):
    """
    Adds second-order interaction features to the dataset.
    
    Args:
        data_numpy (np.ndarray): Input data of shape (N, 2, D).
        
    Returns:
        np.ndarray: Data with second-order features, shape (N, 2, D + D*D).
    """
    n_samples, n_scenarios, n_features = data_numpy.shape
    
    # Create the outer product for each scenario
    # This creates a tensor of shape (N, 2, D, D)
    outer_products = np.einsum('...i,...j->...ij', data_numpy, data_numpy)
    
    # Flatten the interaction term
    interaction_features = outer_products.reshape(n_samples, n_scenarios, -1)
    
    # Concatenate original features with interaction features
    new_data = np.concatenate([data_numpy, interaction_features], axis=2)
    
    return new_data

# =================================================================
# 2. DATA HANDLING (Simplified for Linear Model)
# =================================================================
class MoralMachineLinearDataset(Dataset):
    """
    Custom PyTorch Dataset for the linear model.
    It simply returns the (2, D) tensor and the label.
    """
    def __init__(self, data_matrix, labels):
        self.X = torch.tensor(data_matrix, dtype=torch.float32)
        self.y = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# =================================================================
# 3. MODEL DEFINITION
# =================================================================
class LinearModel(nn.Module):
    """
    A simple Logistic Regression model.
    It takes the difference between two scenarios as input.
    """
    def __init__(self, num_features):
        super().__init__()
        # A single linear layer mapping the difference vector to a single logit
        self.linear = nn.Linear(num_features, 1)

    def forward(self, x):
        # x has shape [batch_size, 2, num_features]
        scenario_a = x[:, 0, :]
        scenario_b = x[:, 1, :]
        
        # Create the "difference vector" which represents the trade-off
        diff = scenario_a - scenario_b
        
        # Pass the difference through the linear layer
        logits = self.linear(diff)
        return logits.squeeze(-1)

# =================================================================
# 4. TRAINING & EVALUATION FUNCTIONS (Mostly unchanged)
# =================================================================
def train_epoch(model, dataloader, optimizer, scheduler, loss_fn, device, epoch):
    model.train()
    total_loss = 0
    for i, (data, labels) in enumerate(tqdm(dataloader, desc=f"Training Epoch {epoch}")):
        data, labels = data.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(data)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate(model, dataloader, loss_fn, device):
    model.eval()
    total_loss, correct_predictions, total_samples = 0, 0, 0
    with torch.no_grad():
        for data, labels in tqdm(dataloader, desc="Evaluating"):
            data, labels = data.to(device), labels.to(device)
            logits = model(data)
            loss = loss_fn(logits, labels)
            preds = (torch.sigmoid(logits) > 0.5).long()
            total_loss += loss.item()
            correct_predictions += (preds == labels.long()).sum().item()
            total_samples += labels.size(0)
    return total_loss / len(dataloader), correct_predictions / total_samples


/opt/miniconda3/envs/dharma/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


: 

In [ ]:

# =================================================================
# 5. MAIN EXECUTION BLOCK
# =================================================================
if __name__ == "__main__":
    if WANDB_AVAILABLE:
        wandb.init(project=hyperparameter_config["project_name"], name=hyperparameter_config["run_name"], config=hyperparameter_config)
        config = wandb.config
    else:
        config = hyperparameter_config
    
    print(f"Using device: {DEVICE}")
    print(f"Running model type: {config.model_type}")

    # --- Load and Process Data ---
    if not os.path.exists(config.data_file_path):
        raise FileNotFoundError(f"Data file not found at '{config.data_file_path}'.")
    
    original_data_tensor = torch.load(config.data_file_path, map_location='cpu')
    original_data_numpy = original_data_tensor.numpy()
    
    print(f"Original data matrix shape: {original_data_numpy.shape}")
    assert original_data_numpy.shape[-1] == BASE_NUM_FEATURES, "Data dimension mismatch"

    # Add second-order features
    print("Adding second-order features...")
    original_data_with_interactions = add_second_order_features(original_data_numpy)
    print(f"Data shape after adding interactions: {original_data_with_interactions.shape}")

    num_original_samples = original_data_with_interactions.shape[0]
    original_labels = np.zeros(num_original_samples)
    swapped_data = original_data_with_interactions[:, [1, 0], :]
    swapped_labels = np.ones(num_original_samples)
    final_data = np.concatenate([original_data_with_interactions, swapped_data], axis=0)
    final_labels = np.concatenate([original_labels, swapped_labels], axis=0)
    print(f"Final balanced data matrix shape: {final_data.shape}")

    # --- Setup Dataset and DataLoaders ---
    dataset = MoralMachineLinearDataset(final_data, final_labels)
    train_size = int(config.train_split_ratio * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    print("Split done")
    
    # NOTE: No custom collate_fn is needed for the linear model
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size)

    # --- Initialize Model, Optimizer, etc. ---
    print("\nInitializing model...")
    model = LinearModel(num_features=NUM_FEATURES).to(DEVICE)
    if WANDB_AVAILABLE:
        wandb.watch(model, log_freq=100)

    optimizer = AdamW(model.parameters(), lr=config.learning_rate)
    total_steps = len(train_loader) * config.epochs
    warmup_steps = int(config.warmup_ratio * total_steps)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    loss_fn = nn.BCEWithLogitsLoss()

    # --- Training Loop ---
    print("\nStarting training...")
    for epoch in range(config.epochs):
        avg_train_loss = train_epoch(model, train_loader, optimizer, scheduler, loss_fn, DEVICE, epoch + 1)
        val_loss, val_accuracy = evaluate(model, val_loader, loss_fn, DEVICE)
        
        print(f"--- End of Epoch {epoch + 1}/{config.epochs} ---")
        print(f"Average Training Loss: {avg_train_loss:.4f}")
        print(f"Validation Loss: {val_loss:.4f} | Validation Accuracy: {val_accuracy:.4f}")
        
        if WANDB_AVAILABLE:
            wandb.log({
                "epoch": epoch + 1,
                "avg_train_loss": avg_train_loss,
                "val_loss": val_loss,
                "val_accuracy": val_accuracy,
                "learning_rate": scheduler.get_last_lr()[0]
            })

    print("\nTraining complete.")
    if WANDB_AVAILABLE:
        wandb.finish()
    
    print("Linear baseline model ready.")


wandb: Currently logged in as: themayankgoel28 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Using device: cpu
Running model type: Linear_2nd_Order
Original data matrix shape: (1749370, 2, 23)
Adding second-order features...
Data shape after adding interactions: (1749370, 2, 552)
Data shape after adding interactions: (1749370, 2, 552)
Final balanced data matrix shape: (3498740, 2, 552)
Final balanced data matrix shape: (3498740, 2, 552)


In [ ]:

# =================================================================
# 6. DIAGNOSTIC CELL
# =================================================================
# Let's check memory and data shape before we proceed, as the kernel crashing
# is often a memory issue.

try:
    import psutil
    
    # --- Check System Memory ---
    memory = psutil.virtual_memory()
    total_gb = memory.total / (1024**3)
    available_gb = memory.available / (1024**3)
    print(f"System Memory: Total={total_gb:.2f} GB, Available={available_gb:.2f} GB")

    # --- Estimate Data Size ---
    if 'final_data' in locals() and 'final_labels' in locals():
        data_bytes = final_data.nbytes
        labels_bytes = final_labels.nbytes
        total_data_gb = (data_bytes + labels_bytes) / (1024**3)
        print(f"Estimated size of the final dataset: {total_data_gb:.2f} GB")
        
        if total_data_gb > available_gb * 0.8:
            print("\n--- WARNING ---")
            print("The dataset size is very large compared to available RAM.")
            print("This is the likely cause of the kernel crash.")
            print("Consider reducing the dataset size, not pre-loading it all into memory, or using a machine with more RAM.")
            print("-----------------")

except ImportError:
    print("psutil not found. Run `pip install psutil` to check memory usage.")
except NameError:
    print("Run the main execution block (cell 2) first to populate data variables.")
